# CS331.R11 — Vehicle Detection & Tracking Experiment

Notebook dùng chung cho **Local** và **Google Colab**.

Chức năng hiện tại: nhận diện môi trường, chuẩn hóa đường dẫn, mount Drive trên Colab, clone repository, cài dependencies, kiểm tra GPU/CUDA và chuẩn bị cấu trúc runtime.


## 1. User configuration

Trên Colab, điền `GITHUB_REPO`. Trên local có thể để trống.


In [ ]:
PROJECT_NAME = "CS331-Vehicle-Tracking"
GITHUB_REPO = "https://github.com/zaabinh/CS331-Vehicle-Tracking.git"  
GITHUB_BRANCH = "main"
DRIVE_PROJECT_DIR = f"/content/drive/MyDrive/{PROJECT_NAME}"
AUTO_INSTALL_REQUIREMENTS = True

print("Configuration loaded.")


## 2. Nhận diện môi trường


In [ ]:
import os
import sys
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

RUNNING_IN_COLAB = is_colab()
ENVIRONMENT = "COLAB" if RUNNING_IN_COLAB else "LOCAL"

print("Environment:", ENVIRONMENT)
print("Python:", sys.version.split()[0])
print("CWD:", Path.cwd())


## 3. Google Drive — chỉ mount trên Colab


In [ ]:
DRIVE_ROOT = None

if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_ROOT = Path(DRIVE_PROJECT_DIR)
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    for folder in ["weights", "detections", "experiments", "outputs", "checkpoints"]:
        (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

    print("Drive root:", DRIVE_ROOT)
else:
    print("Local environment: skip Google Drive.")


## 4. Xác định repository

- Colab: clone GitHub vào `/content`.
- Local: tìm project root từ `.git` hoặc `requirements.txt`.


In [ ]:
import subprocess

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists():
            return p
    return start.parent if start.name == "notebooks" else start

if RUNNING_IN_COLAB:
    PROJECT_ROOT = Path("/content") / PROJECT_NAME

    if not PROJECT_ROOT.exists():
        if not GITHUB_REPO.strip():
            raise ValueError("Hãy điền GITHUB_REPO trước khi chạy trên Colab.")

        subprocess.run(
            ["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(PROJECT_ROOT)],
            check=True
        )
    else:
        print("Repository already exists:", PROJECT_ROOT)
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)


## 5. Runtime directories


In [ ]:
DATA_DIR = Path("/content/data") if RUNNING_IN_COLAB else PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CACHE_DIR = PROJECT_ROOT / "detections"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

for p in [DATA_DIR, WEIGHTS_DIR, OUTPUT_DIR, CACHE_DIR, EXPERIMENTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR       :", DATA_DIR)
print("WEIGHTS_DIR    :", WEIGHTS_DIR)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("CACHE_DIR      :", CACHE_DIR)
print("EXPERIMENTS_DIR:", EXPERIMENTS_DIR)


## 6. Cài dependencies từ `requirements.txt`


In [ ]:
requirements = PROJECT_ROOT / "requirements.txt"

if AUTO_INSTALL_REQUIREMENTS and requirements.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True
    )
    print("Dependencies installed.")
elif not requirements.exists():
    print("requirements.txt not found — skip.")
else:
    print("Dependency installation disabled.")


## 7. Kiểm tra PyTorch / CUDA / OpenCV / Ultralytics


In [ ]:
import platform

print("=" * 55)
print("Environment:", ENVIRONMENT)
print("OS:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        DEVICE = 0
    else:
        DEVICE = "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
    print("PyTorch: NOT INSTALLED")

try:
    import cv2
    print("OpenCV:", cv2.__version__)
except ImportError:
    print("OpenCV: NOT INSTALLED")

try:
    import ultralytics
    print("Ultralytics:", ultralytics.__version__)
except ImportError:
    print("Ultralytics: NOT INSTALLED")

print("Selected device:", DEVICE)


## 8. Chuẩn bị `best.pt`

- Local: đặt tại `weights/best.pt`.
- Colab: có thể lưu persistent tại `MyDrive/CS331-Vehicle-Tracking/weights/best.pt`; notebook sẽ copy sang runtime.


In [ ]:
import shutil

MODEL_NAME = "best.pt"
LOCAL_MODEL_PATH = WEIGHTS_DIR / MODEL_NAME

if RUNNING_IN_COLAB and DRIVE_ROOT is not None:
    DRIVE_MODEL_PATH = DRIVE_ROOT / "weights" / MODEL_NAME
    if not LOCAL_MODEL_PATH.exists() and DRIVE_MODEL_PATH.exists():
        shutil.copy2(DRIVE_MODEL_PATH, LOCAL_MODEL_PATH)
        print("Copied:", DRIVE_MODEL_PATH, "->", LOCAL_MODEL_PATH)

if LOCAL_MODEL_PATH.exists():
    print("Model found:", LOCAL_MODEL_PATH)
else:
    print("best.pt chưa có; setup vẫn có thể tiếp tục.")


## 9. Experiment configuration


In [ ]:
EXPERIMENT_CONFIG = {
    "dataset": "UA-DETRAC",
    "detector": "YOLOv8",
    "model_path": str(LOCAL_MODEL_PATH),
    "trackers": ["SORT", "DeepSORT", "ByteTrack"],
    "device": DEVICE,
    "save_video": True,
    "cache_detections": True,
}

EXPERIMENT_CONFIG


## 10. Smoke test YOLO


In [ ]:
if LOCAL_MODEL_PATH.exists():
    from ultralytics import YOLO
    model = YOLO(str(LOCAL_MODEL_PATH))
    print("YOLO model loaded successfully.")
else:
    model = None
    print("Skip: best.pt not available.")


## 11. Project readiness


In [ ]:
checks = {
    "Project root": PROJECT_ROOT.exists(),
    "Data directory": DATA_DIR.exists(),
    "Weights directory": WEIGHTS_DIR.exists(),
    "Output directory": OUTPUT_DIR.exists(),
    "Detection cache": CACHE_DIR.exists(),
    "Experiments directory": EXPERIMENTS_DIR.exists(),
}

for name, status in checks.items():
    print(f"{'OK' if status else 'FAILED':6} | {name}")

print("-" * 55)
print("Base environment ready." if all(checks.values()) else "Setup has failed checks.")
